# ENPIRE VLA Autoresearch — Kaggle Setup

Run on **Kaggle GPU T4 x2** (uses one 16GB T4 via 4-bit OpenVLA).

1. Settings → Accelerator → **GPU T4 x2**
2. Add `OPENAI_API_KEY` in **Add-ons → Secrets**
3. Enable **Internet**

**Weekly limit:** ~30 GPU hours/week.

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/enpire-vla-autoresearch"
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/EdnilsonChiambo/enpire-vla-autoresearch.git", REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
except Exception:
    print("Warning: set OPENAI_API_KEY manually if not using Kaggle Secrets")

print("Working directory:", os.getcwd())
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
print("Kaggle deps installed.")

## Option A: Quick loop test (Gym-PushT + mock VLA)

Validates the autoresearch loop without SIMPLER (~1 min).

In [ ]:
os.environ["VLA_BACKEND"] = "mock"
os.environ["ENV_BACKEND"] = "pusht"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gym-pusht", "pymunk>=6.6.0,<7.0.0"], check=True)
subprocess.run([sys.executable, "-m", "src.loop_runner", "--dry-run"], check=True)

## Option B: SIMPLER + OpenVLA 4-bit (full VLA path)

First run may take 30–60 min (SIMPLER install + model download). Uses ~6–7GB VRAM for the model.

In [ ]:
import shutil

REPO_DIR = "/kaggle/working/enpire-vla-autoresearch"
SIMPLER_DIR = "/kaggle/working/SimplerEnv"

subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# ManiSkill3 branch for Python 3.12 (main needs sapien 2.2.2).
# Shallow main clone cannot checkout `maniskill3` by name — re-clone if needed.
if os.path.isdir(SIMPLER_DIR):
    result = subprocess.run(
        ["git", "-C", SIMPLER_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0 or result.stdout.strip() != "maniskill3":
        shutil.rmtree(SIMPLER_DIR)

if not os.path.isdir(SIMPLER_DIR):
    subprocess.run(
        [
            "git", "clone", "-b", "maniskill3", "--depth", "1",
            "https://github.com/simpler-env/SimplerEnv.git", SIMPLER_DIR,
        ],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/haosulab/ManiSkill.git", "tyro==0.8.5"],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", SIMPLER_DIR], check=True)

# Fresh subprocess sees editable installs; this Jupyter kernel may not until restart.
verify = subprocess.run(
    [sys.executable, "-c", "import simpler_env; print(simpler_env.__file__)"],
    capture_output=True,
    text=True,
    check=True,
)
print("pip editable install OK:", verify.stdout.strip())

if SIMPLER_DIR not in sys.path:
    sys.path.insert(0, SIMPLER_DIR)

import gymnasium as gym
import mani_skill
import simpler_env
print("SIMPLER ManiSkill3 OK:", simpler_env.__file__)

os.environ["VLA_BACKEND"] = "openvla-4bit"
os.environ["ENV_BACKEND"] = "simpler"
os.environ["SIMPLER_BACKEND"] = "maniskill3"
os.environ["SIMPLER_DIR"] = SIMPLER_DIR
print("SIMPLER ready at", SIMPLER_DIR)

In [ ]:
from src.vla.openvla_adapter_4bit import OpenVLA4BitAdapter
import numpy as np

adapter = OpenVLA4BitAdapter(policy_setup="bridge_orig")
action = adapter.predict_action(np.zeros((224, 224, 3), dtype=np.uint8), "pick up the spoon")
print("OpenVLA 4-bit OK. Action shape:", action.shape)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "src.loop_runner", "--config", "config/kaggle.yaml"],
    check=True,
)

In [ ]:
from pathlib import Path
for path in ["logs/latest_eval_log.txt", "logs/history_tree.log", "logs/best_run.json"]:
    p = Path(path)
    if p.exists():
        print(f"\n=== {path} ===\n{p.read_text()[:2000]}")